In [ ]:
#reset all variables
%reset -f

In [ ]:
from sionna.rt import load_scene, PlanarArray, Transmitter, Receiver, RadioMapSolver, ITURadioMaterial, PathSolver
import mitsuba as mi
import numpy as np

#Load scene, assing frequency and
scene = load_scene("room/room.xml",merge_shapes=False)
scene.frequency = 29.5e9 #Hz
scene.synthetic_array = True

#See objects and materials from blender
# for name, obj in scene.objects.items():
#     print(f'{name:<15}{obj.radio_material.name}')

#Apply material and thickness
matAndThick = ITURadioMaterial(name="plasterboard",
                               itu_type="plasterboard",
                               thickness=0.03)#m
obj = scene.get("P2")
obj.radio_material = matAndThick
obj = scene.get("P1")
obj.radio_material = matAndThick

#Configure antennas arrays
scene.tx_array = PlanarArray(
   num_rows=1,
   num_cols=4,
   vertical_spacing=0.5,
   horizontal_spacing=0.5,
   pattern="iso",
   polarization="V")
scene.rx_array = PlanarArray(
    num_rows=1,
    num_cols=1,
    vertical_spacing=0.5,
    horizontal_spacing=0.5,
    pattern="dipole",
    polarization="V")

# Transmitter
tx = Transmitter(name="tx",
                 position=mi.Point3f(0, 0, 1.2),
                 orientation=mi.Point3f(0, 0, 0),
                 power_dbm=37)
scene.add(tx)

# Receiver
rx = Receiver(name="rx",
              position=mi.Point3f(-3,5,1.2),
              orientation=(0.0, 0.0, 0.0))
scene.add(rx)
rx = Receiver(name="rx2",
              position=mi.Point3f(3,5,1.2),
              orientation=(0.0, 0.0, 0.0))
scene.add(rx)


#Paths
solver  = PathSolver()
paths = solver(scene,
               max_depth=6,
               max_num_paths_per_src= int(1e6),
               samples_per_src= int(1e6),
               synthetic_array=True,
               los=True,
               specular_reflection=True,
               diffuse_reflection=True,
               refraction=True,
               edge_diffraction=True,
               diffraction_lit_region=True)

# Radio Map
rmSolver = RadioMapSolver()
rm = rmSolver(scene,
              samples_per_tx= int(1e6),
              max_depth=6,
              cell_size=(0.1, 0.1),
              los = True,
              specular_reflection = True,
              diffuse_reflection = True,
              refraction = True,
              diffraction = True,
              edge_diffraction = True,
              diffraction_lit_region = True,
              stop_threshold = -131)

#Ampltiudes and TDoA from CIR
a, tau = paths.cir(out_type="numpy")

print("rx, rx_ant, tx, tx_ant, paths, batch")
print(a.shape)
print("rxs, txs, paths")
print(tau.shape)

nReceptor = 2
tx_power_dbm = 37

print("\n--- RSS ---")

for rx in range(nReceptor):
    amp = a[rx, 0, 0, :, :, 0]
    power = np.sum(np.abs(amp)**2)
    dBPower = tx_power_dbm + 10*np.log10(power + 1e-15)
    print(f"\n[Receptor {rx}] tiene RSS {dBPower:.2f} dBm")



scene.preview(radio_map=rm, clip_at=3)
rm.show(metric='path_gain', vmin=-131)
rm.show(metric='rss', vmin=-100, vmax=-40)

In [ ]:
from sionna.rt import load_scene, PlanarArray, Transmitter, Receiver, RadioMapSolver, ITURadioMaterial, PathSolver
import mitsuba as mi
import numpy as np
import pandas as pd

scene = load_scene("scenev3/final_scene_v3.xml",merge_shapes=False)
scene.frequency = 29.5e9 #Hz
scene.synthetic_array = True

tx_power_dbm = 33.5

#Create material and thickness
Thick_Plasterboard = ITURadioMaterial(name="plasterboard",
                               itu_type="plasterboard",
                               thickness=0.09)#m

Thick_Brick = ITURadioMaterial(name="brick",
                               itu_type="brick",
                               thickness=0.3)#m

Thick_Glass = ITURadioMaterial(name="glass",
                               itu_type="glass",
                               thickness=0.3)#m

#See objects and materials from blender
for name, obj in scene.objects.items():
    print(f'\033[1m{name:}---{obj.radio_material.name}\033[0m')
    if obj.radio_material.name == "itu_plasterboard":
        obj.radio_material = Thick_Plasterboard #Applies thickness
        print("    -Thickness to plasterboard applied")
    if obj.radio_material.name == "itu_brick":
        obj.radio_material = Thick_Brick #Applies thickness
        print("    -Thickness to brick applied")
    if obj.radio_material.name == "itu_glass":
        obj.radio_material = Thick_Glass #Applies thickness
        print("    -Thickness to glass applied")

# Configure antennas arrays
scene.tx_array = PlanarArray( #Transmitter
    num_rows=1,
    num_cols=4,
    vertical_spacing=0.5,
    horizontal_spacing=0.5,
    pattern="iso",
    polarization = "V")

scene.rx_array = PlanarArray( #Receiver
    num_rows=1,
    num_cols=1,
    vertical_spacing=0.5,
    horizontal_spacing=0.5,
    pattern="dipole",
    polarization="V")

#Create transmitter and receiver
tx = Transmitter(name="tx",
                 position=mi.Point3f(-7, 0, 1),
                 orientation=mi.Point3f(0, 0, 0),
                 power_dbm= tx_power_dbm)
scene.add(tx)
tx.look_at([-6, 0, 2])
numreceptors = ["1", "2", "3", "4", "5", "6", "7", "7b", "8", "9", "10", "11", "12", "13", "14", "15", "16", "17", "17b", "18", "19", "20"]
nReceptor = 22
positions = ( (-6.5 , 4),   #1
              (-4   , 3),   #2
              (-2   , 3),   #3
              (0    , 3),   #4
              (1    , 1.2), #5
              (-2   , 1.2), #6
              (-6   , 1.2), #7
              (-4.5 , 1.2), #7b
              (-5.3 , -0.5),#8
              (-2.5 , 0.2), #9
              (0.5  , 0.5), #10
              (2.5  , 1),   #11
              (3    , -1),  #12
              (1    , -2),  #13
              (-2   , -2),  #14
              (-5.3 , -2.5),#15
              (-7   , -2.5),#16
              (-5.7 , -4),  #17
              (-4   , -4),  #17b
              (-2   , -4),  #18
              (1    , -4),  #19
              (3    , -4))  #20
for i in range(nReceptor):
    posx = positions[i][0]
    posy = positions[i][1]
    posz = 1
    rx = Receiver(name=f"rx_{i}",
                 position=mi.Point3f(posx, posy, posz),
                 orientation=mi.Point3f(0.0, 0.0, 0.0))
    scene.add(rx)

# Radio Map
rmSolver = RadioMapSolver()
rm = rmSolver(scene,
              samples_per_tx= int(1e6),
              max_depth=6,
              cell_size=(0.1, 0.1),
              los = True,
              specular_reflection = True,
              diffuse_reflection = True,
              refraction = True,
              diffraction = True,
              edge_diffraction = True,
              diffraction_lit_region = True,
              stop_threshold = -131)

#Paths
solver  = PathSolver()
paths = solver(scene,
               max_depth=6,
               max_num_paths_per_src= int(1e6),
               samples_per_src= int(1e6),
               synthetic_array=True,
               los=True,
               specular_reflection=True,
               diffuse_reflection=True,
               refraction=True,
               edge_diffraction=False,
               diffraction_lit_region=True)

#scene.preview(radio_map=rm, clip_at=3)
# rm.show(metric='path_gain', vmin=-131)
rm.show(metric='rss', vmin=-100, vmax=-5)

#Ampltiudes and TDoA from CIR
a, tau = paths.cir(out_type="numpy")

# print("rx, rx_ant, tx, tx_ant, paths, batch")
# print(a.shape)
# print("rxs, txs, paths")
# print(tau.shape)

SimPowerdB = np.zeros(22)

print("\n--- RSS ---")

for rx in range(nReceptor):
    amp = a[rx, 0, 0, :, :, 0]
    power = np.sum(np.abs(amp)**2)
    dBPower = tx_power_dbm + 10*np.log10(power + 1e-15)
    SimPowerdB[rx] = dBPower
    #print(f"\nReceptor {numreceptors[rx]} ubicado en {positions[rx]}tiene RSS {dBPower:.2f} dBm")

df = pd.read_csv("Resultados.csv")
data = df["90"].to_numpy()

for i in range(nReceptor):
    d1 = data[i]
    d2 = SimPowerdB[i]
    r = np.abs(d1) - np.abs(d2)
    print(f"Difference in receptor {numreceptors[i]} is: {r:.2f} dBm" )
    if (np.abs(r) <= 4):
        print('----> Valid')


In [1]:
from sionna.rt import load_scene, PlanarArray, Transmitter, Receiver, RadioMapSolver, ITURadioMaterial, PathSolver
import mitsuba as mi
import numpy as np

#Load scene, assing frequency and
scene = load_scene("room/room.xml",merge_shapes=False)
scene.frequency = 29.5e9 #Hz
scene.synthetic_array = True

#See objects and materials from blender
# for name, obj in scene.objects.items():
#     print(f'{name:<15}{obj.radio_material.name}')

#Apply material and thickness
matAndThick = ITURadioMaterial(name="plasterboard",
                               itu_type="plasterboard",
                               thickness=0.03)#m
obj = scene.get("P2")
obj.radio_material = matAndThick
obj = scene.get("P1")
obj.radio_material = matAndThick

#Configure antennas arrays
scene.tx_array = PlanarArray(
   num_rows=1,
   num_cols=4,
   vertical_spacing=0.5,
   horizontal_spacing=0.5,
   pattern="iso",
   polarization="V")
scene.rx_array = PlanarArray(
    num_rows=1,
    num_cols=1,
    vertical_spacing=0.5,
    horizontal_spacing=0.5,
    pattern="dipole",
    polarization="V")


jitc_llvm_init(): LLVM API initialization failed ..
